# Conferência Ressarcimento — ADMStoIQS

Objetivo: auditar por que o ressarcimento estimado está em aproximadamente `R$ 12 milhões`, quando o esperado é próximo de `R$ 7 milhões`.

O notebook decompõe o cálculo por:

- cenário `antes` / `depois`;
- indicador que venceu por UC: DIC, FIC ou DMIC;
- grupo de tensão e KEI;
- VRC;
- metas;
- duplicidade de UC;
- diferença entre fórmula atual e variações alternativas.

Use principalmente as seções 3, 4, 5 e 8 para localizar o desvio.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path(r"D:\ADMStoIQS")
ANOMES = "202605"

indicadores_uc = ROOT / "data" / "mart" / "indicadores" / f"indicadores_uc_{ANOMES}.parquet"
ressarcimento = ROOT / "data" / "mart" / "indicadores" / f"indicadores_ressarcimento_{ANOMES}.parquet"
ressarcimento_atual = ROOT / "data" / "mart" / "indicadores" / "indicadores_ressarcimento_ATUAL.parquet"
metas_uc = ROOT / "data" / "external" / "iqs" / "raw" / f"metas_uc_{ANOMES}.parquet"
vrc = ROOT / "data" / "external" / "iqs" / "raw" / f"vrc_{ANOMES}.parquet"

paths = {
    "indicadores_uc": indicadores_uc,
    "ressarcimento": ressarcimento,
    "ressarcimento_atual": ressarcimento_atual,
    "metas_uc": metas_uc,
    "vrc": vrc,
}

pd.DataFrame([{"arquivo": k, "existe": v.exists(), "caminho": str(v)} for k, v in paths.items()])

## 1. Ler resultado materializado

Esta é a visão do arquivo oficial gerado pelo backend.

In [ ]:
con = duckdb.connect(database=":memory:")

def describe(path: Path):
    if not path.exists():
        return pd.DataFrame()
    return con.execute("DESCRIBE SELECT * FROM read_parquet(?)", [str(path)]).df()

display(describe(ressarcimento))

resumo_materializado = con.execute(
    """
    SELECT
        cenario,
        COUNT(*) AS linhas,
        COUNT(DISTINCT uc) AS ucs,
        SUM(CASE WHEN possui_violacao THEN 1 ELSE 0 END) AS linhas_com_violacao,
        SUM(compensacao_dic) AS soma_comp_dic,
        SUM(compensacao_fic) AS soma_comp_fic,
        SUM(compensacao_dmic) AS soma_comp_dmic,
        SUM(valor_ressarcimento_estimado) AS valor_total
    FROM read_parquet(?)
    GROUP BY cenario
    ORDER BY cenario
    """,
    [str(ressarcimento)],
).df()
resumo_materializado

## 2. Quem está vencendo: DIC, FIC ou DMIC?

Pelo PRODIST operacional usado aqui, soma-se o maior valor individual da UC. Esta seção mostra qual indicador está comandando o total.

In [ ]:
vencedor_df = con.execute(
    """
    WITH base AS (
        SELECT
            *,
            CASE
                WHEN valor_ressarcimento_estimado = 0 THEN 'SEM_VIOLACAO'
                WHEN compensacao_dic >= compensacao_fic AND compensacao_dic >= compensacao_dmic THEN 'DIC'
                WHEN compensacao_fic >= compensacao_dic AND compensacao_fic >= compensacao_dmic THEN 'FIC'
                ELSE 'DMIC'
            END AS indicador_vencedor
        FROM read_parquet(?)
    )
    SELECT
        cenario,
        indicador_vencedor,
        COUNT(*) AS linhas,
        COUNT(DISTINCT uc) AS ucs,
        SUM(valor_ressarcimento_estimado) AS valor_total,
        AVG(valor_ressarcimento_estimado) AS valor_medio,
        MAX(valor_ressarcimento_estimado) AS maior_valor
    FROM base
    GROUP BY cenario, indicador_vencedor
    ORDER BY cenario, valor_total DESC
    """,
    [str(ressarcimento)],
).df()
vencedor_df

## 3. Top UCs por valor

Se poucos registros explicarem boa parte dos R$ 12 milhões, pode haver VRC alto, duração/DMIC alta ou meta muito baixa.

In [ ]:
top_ucs = con.execute(
    """
    SELECT
        cenario,
        uc,
        regional,
        conjunto_aneel,
        grupo_tensao,
        nivel_tensao,
        urb_rur,
        vrc,
        kei,
        dic_horas,
        limite_dic_horas,
        fic,
        limite_fic,
        dmic_horas,
        limite_dmic_horas,
        compensacao_dic,
        compensacao_fic,
        compensacao_dmic,
        valor_ressarcimento_estimado
    FROM read_parquet(?)
    WHERE cenario = 'depois'
      AND valor_ressarcimento_estimado > 0
    ORDER BY valor_ressarcimento_estimado DESC
    LIMIT 100
    """,
    [str(ressarcimento)],
).df()
top_ucs

## 4. Distribuição por grupo de tensão / KEI

O KEI multiplica diretamente o valor. Conferir se há UCs com grupo/nivel incorretos é essencial.

In [ ]:
grupo_kei = con.execute(
    """
    SELECT
        cenario,
        COALESCE(grupo_tensao, 'SEM_GRUPO') AS grupo_tensao,
        COALESCE(nivel_tensao, 'SEM_NIVEL') AS nivel_tensao,
        kei,
        COUNT(*) AS linhas,
        COUNT(DISTINCT uc) AS ucs,
        SUM(valor_ressarcimento_estimado) AS valor_total,
        AVG(vrc) AS vrc_medio,
        MAX(vrc) AS vrc_maximo
    FROM read_parquet(?)
    WHERE valor_ressarcimento_estimado > 0
    GROUP BY cenario, COALESCE(grupo_tensao, 'SEM_GRUPO'), COALESCE(nivel_tensao, 'SEM_NIVEL'), kei
    ORDER BY cenario, valor_total DESC
    """,
    [str(ressarcimento)],
).df()
grupo_kei

## 5. Auditoria de duplicidade por UC/cenário

O arquivo de ressarcimento deve ter uma linha por UC por cenário. Se houver duplicidade, o total pode inflar.

In [ ]:
duplicidade = con.execute(
    """
    WITH dup AS (
        SELECT cenario, uc, COUNT(*) AS qtd, SUM(valor_ressarcimento_estimado) AS valor
        FROM read_parquet(?)
        GROUP BY cenario, uc
        HAVING COUNT(*) > 1
    )
    SELECT
        cenario,
        COUNT(*) AS ucs_duplicadas,
        SUM(qtd) AS linhas_duplicadas,
        SUM(valor) AS valor_em_duplicidade
    FROM dup
    GROUP BY cenario
    ORDER BY cenario
    """,
    [str(ressarcimento)],
).df()
duplicidade

In [ ]:
duplicadas_detalhe = con.execute(
    """
    SELECT *
    FROM read_parquet(?)
    WHERE (cenario, uc) IN (
        SELECT cenario, uc
        FROM read_parquet(?)
        GROUP BY cenario, uc
        HAVING COUNT(*) > 1
    )
    ORDER BY cenario, uc, valor_ressarcimento_estimado DESC
    LIMIT 100
    """,
    [str(ressarcimento), str(ressarcimento)],
).df()
duplicadas_detalhe

## 6. Conferir VRC

Valores de VRC muito altos podem explicar excesso. A consulta abaixo mostra distribuição e top UCs por VRC.

In [ ]:
vrc_stats = con.execute(
    """
    SELECT
        cenario,
        COUNT(*) AS linhas,
        COUNT(DISTINCT uc) AS ucs,
        MIN(vrc) AS vrc_min,
        AVG(vrc) AS vrc_avg,
        quantile_cont(vrc, 0.5) AS vrc_p50,
        quantile_cont(vrc, 0.9) AS vrc_p90,
        quantile_cont(vrc, 0.99) AS vrc_p99,
        MAX(vrc) AS vrc_max
    FROM read_parquet(?)
    GROUP BY cenario
    ORDER BY cenario
    """,
    [str(ressarcimento)],
).df()
vrc_stats

In [ ]:
top_vrc = con.execute(
    """
    SELECT
        cenario,
        uc,
        regional,
        grupo_tensao,
        nivel_tensao,
        vrc,
        kei,
        valor_ressarcimento_estimado
    FROM read_parquet(?)
    ORDER BY vrc DESC NULLS LAST
    LIMIT 100
    """,
    [str(ressarcimento)],
).df()
top_vrc

## 7. Conferir metas zeradas/nulas

Metas nulas ou zeradas podem gerar interpretação errada. O serviço atual só considera violação quando o realizado é maior que a meta disponível; aqui auditamos os casos nulos/zeros.

In [ ]:
metas_stats = con.execute(
    """
    SELECT
        cenario,
        SUM(CASE WHEN limite_dic_horas IS NULL THEN 1 ELSE 0 END) AS meta_dic_nula,
        SUM(CASE WHEN limite_fic IS NULL THEN 1 ELSE 0 END) AS meta_fic_nula,
        SUM(CASE WHEN limite_dmic_horas IS NULL THEN 1 ELSE 0 END) AS meta_dmic_nula,
        SUM(CASE WHEN limite_dic_horas = 0 THEN 1 ELSE 0 END) AS meta_dic_zero,
        SUM(CASE WHEN limite_fic = 0 THEN 1 ELSE 0 END) AS meta_fic_zero,
        SUM(CASE WHEN limite_dmic_horas = 0 THEN 1 ELSE 0 END) AS meta_dmic_zero,
        SUM(valor_ressarcimento_estimado) AS valor_total
    FROM read_parquet(?)
    GROUP BY cenario
    ORDER BY cenario
    """,
    [str(ressarcimento)],
).df()
metas_stats

## 8. Fórmulas alternativas para comparação

A fórmula atual usa o realizado total, conforme SQL fornecido:

```text
VRC * (REALIZADO / 730) * KEI
```

Mas alguns processos calculam sobre o excedente:

```text
VRC * ((REALIZADO - META) / 730) * KEI
```

Esta seção compara as duas. Se o esperado de R$ 7 mi aparecer na alternativa por excedente, encontramos a diferença conceitual.

In [ ]:
formula_comparada = con.execute(
    """
    WITH base AS (
        SELECT
            *,
            CASE WHEN dic_horas > limite_dic_horas THEN vrc * (dic_horas / 730.0) * kei ELSE 0 END AS atual_dic,
            CASE WHEN fic > limite_fic THEN vrc * (fic / 730.0) * kei ELSE 0 END AS atual_fic,
            CASE WHEN dmic_horas > limite_dmic_horas THEN vrc * (dmic_horas / 730.0) * kei ELSE 0 END AS atual_dmic,
            CASE WHEN dic_horas > limite_dic_horas THEN vrc * ((dic_horas - limite_dic_horas) / 730.0) * kei ELSE 0 END AS excedente_dic,
            CASE WHEN fic > limite_fic THEN vrc * ((fic - limite_fic) / 730.0) * kei ELSE 0 END AS excedente_fic,
            CASE WHEN dmic_horas > limite_dmic_horas THEN vrc * ((dmic_horas - limite_dmic_horas) / 730.0) * kei ELSE 0 END AS excedente_dmic
        FROM read_parquet(?)
    ),
    calc AS (
        SELECT
            *,
            GREATEST(atual_dic, atual_fic, atual_dmic) AS valor_formula_atual,
            GREATEST(excedente_dic, excedente_fic, excedente_dmic) AS valor_formula_excedente
        FROM base
    )
    SELECT
        cenario,
        SUM(valor_formula_atual) AS total_formula_atual,
        SUM(valor_formula_excedente) AS total_formula_excedente,
        SUM(valor_formula_atual) - SUM(valor_formula_excedente) AS diferenca,
        COUNT(DISTINCT CASE WHEN valor_formula_atual > 0 THEN uc END) AS ucs_compensadas_atual,
        COUNT(DISTINCT CASE WHEN valor_formula_excedente > 0 THEN uc END) AS ucs_compensadas_excedente
    FROM calc
    GROUP BY cenario
    ORDER BY cenario
    """,
    [str(ressarcimento)],
).df()
formula_comparada

## 9. Alternativa com DIC/DMIC em horas e FIC como quantidade

A fórmula fornecida divide também FIC por 730. Esta seção existe só para evidência: se houver regra interna diferente para FIC, compare aqui antes de alterar o backend.

In [ ]:
fic_alternativo = con.execute(
    """
    WITH base AS (
        SELECT
            *,
            CASE WHEN dic_horas > limite_dic_horas THEN vrc * ((dic_horas - limite_dic_horas) / 730.0) * kei ELSE 0 END AS comp_dic_excedente,
            CASE WHEN fic > limite_fic THEN vrc * ((fic - limite_fic) / 730.0) * kei ELSE 0 END AS comp_fic_excedente_div730,
            CASE WHEN dmic_horas > limite_dmic_horas THEN vrc * ((dmic_horas - limite_dmic_horas) / 730.0) * kei ELSE 0 END AS comp_dmic_excedente,
            CASE WHEN fic > limite_fic THEN vrc * (fic - limite_fic) * kei ELSE 0 END AS comp_fic_sem_div730
        FROM read_parquet(?)
    )
    SELECT
        cenario,
        SUM(GREATEST(comp_dic_excedente, comp_fic_excedente_div730, comp_dmic_excedente)) AS total_excedente_fic_div730,
        SUM(GREATEST(comp_dic_excedente, comp_fic_sem_div730, comp_dmic_excedente)) AS total_excedente_fic_sem_div730
    FROM base
    GROUP BY cenario
    ORDER BY cenario
    """,
    [str(ressarcimento)],
).df()
fic_alternativo

## 10. Checklist de diagnóstico

- Se a seção 5 mostrar duplicidade, corrigir o serviço para consolidar uma linha por UC/cenário.
- Se a seção 8 aproximar `total_formula_excedente` de R$ 7 mi, a regra esperada usa excedente sobre meta, não realizado total.
- Se a seção 4 concentrar muito valor em KEI 108, conferir grupo/nível de tensão do VRC.
- Se a seção 6 mostrar VRCs extremos, conferir `VAL_BASE_CALC_COMPEN_UC`.
- Se a seção 7 mostrar muitas metas nulas/zero, conferir extração `metas_uc`.
- Se o valor do cenário `depois` estiver alto por DMIC, investigar registros com `dmic_horas` muito acima da meta.